In [ ]:
import os, copy, torch, torch.nn as nn, torch.optim as optim
import numpy as np
from torchvision import models
from torch.utils.data import DataLoader
from data_preprocessing import get_data_loaders
from evaluation_metrics import evaluate_model_metrics, measure_inference_metrics,measure_model_size_and_flops

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
data_dir = "tiny-imagenet-200"
train_loader, val_loader = get_data_loaders(data_dir, batch_size=64, num_workers=4, image_size=224)
print("len(train_loader):", len(train_loader))
print("len(val_loader):", len(val_loader))

In [ ]:
def build_baseline_model(num_classes=200):
    model = models.mobilenet_v2(pretrained=True)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)
    return model

baseline = build_baseline_model()
baseline = baseline.to(device)
print("Baseline MobileNetV2 model built.")

In [ ]:
def train_model(model, train_loader, num_epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
    return model

print("Starting training baseline model...")
baseline = train_model(baseline, train_loader, num_epochs=10, lr=0.001)
print("Training complete.")

In [ ]:
def evaluate_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    accuracy = 100.0 * correct / total
    return accuracy

acc_fp32 = evaluate_model(baseline, val_loader)
print(f"Baseline Model Validation Accuracy: {acc_fp32:.2f}%")

In [ ]:
def prune_pair(curr: nn.Conv2d,
               nxt: nn.Conv2d,
               prune_ratio: float = 0.2):
    """Prune `prune_ratio` of *output* channels in `curr` and matching *input*
    channels in `nxt`. Returns (new_curr, new_nxt)."""
    w = curr.weight.data.cpu().numpy()                 # (out, in, k, k)
    l1 = np.abs(w).sum(axis=(1, 2, 3))                 # importance per out‑ch
    k  = int(w.shape[0] * (1 - prune_ratio))
    keep_idx = np.argsort(l1)[-k:]
    keep_idx.sort()

    # rebuild current conv (out‑channels pruned)
    new_curr = nn.Conv2d(curr.in_channels, len(keep_idx),
                         kernel_size=curr.kernel_size,
                         stride=curr.stride,
                         padding=curr.padding,
                         dilation=curr.dilation,
                         groups=curr.groups,
                         bias=curr.bias is not None)
    new_curr.weight.data = curr.weight.data[keep_idx].clone()
    if curr.bias is not None:
        new_curr.bias.data = curr.bias.data[keep_idx].clone()

    # rebuild next conv (input‑channels pruned)
    new_nxt = nn.Conv2d(len(keep_idx), nxt.out_channels,
                        kernel_size=nxt.kernel_size,
                        stride=nxt.stride,
                        padding=nxt.padding,
                        dilation=nxt.dilation,
                        groups=nxt.groups,
                        bias=nxt.bias is not None)
    new_nxt.weight.data = nxt.weight.data[:, keep_idx].clone()
    if nxt.bias is not None:
        new_nxt.bias.data = nxt.bias.data.clone()

    return new_curr, new_nxt

In [ ]:
def prune_mobilenet_round(model: nn.Module, prune_ratio: float = 0.2):
    """One pass over MobileNetV2, pruning (dw, pw) pairs in each InvertedResidual."""
    m = copy.deepcopy(model)
    for blk in m.features:
        if isinstance(blk, nn.InvertedResidual):
            dw = blk.conv[0]
            pw = blk.conv[1]
            new_dw, new_pw = prune_pair(dw, pw, prune_ratio)
            blk.conv[0], blk.conv[1] = new_dw, new_pw
    return m

In [ ]:
def iterative_prune_and_finetune(base_model: nn.Module,
                                 train_loader: DataLoader,
                                 val_loader: DataLoader,
                                 device: torch.device,
                                 total_prune: float = 0.3,
                                 rounds: int = 3,
                                 epochs_per_round: int = 3,
                                 lr: float = 1e-3):
    """Prune‑finetune loop: splits `total_prune` evenly across `rounds`."""
    per_round = 1 - (1 - total_prune) ** (1 / rounds)
    model = copy.deepcopy(base_model).to(device)
    criterion = nn.CrossEntropyLoss()

    for r in range(rounds):
        print(f"\n=== Round {r+1}/{rounds} | pruning {per_round*100:.1f}% of channels ===")
        model = prune_mobilenet_round(model, per_round).to(device)

        # fine‑tune
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        model.train()
        for ep in range(epochs_per_round):
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                criterion(model(x), y).backward()
                opt.step()
        acc = evaluate_model_metrics(model, val_loader)
        print(f"  ► Val accuracy after round {r+1}: {acc:.2f}%")
    return model

In [ ]:
pruned_model = iterative_prune_and_finetune(baseline,
                                                train_loader,
                                                val_loader,
                                                device,
                                                total_prune=0.4,  # 40 % overall
                                                rounds=4,
                                                epochs_per_round=2,
                                                lr=1e-3)

In [ ]:
acc = evaluate_model_metrics(pruned_model, val_loader)
print(f"\nCustom Channel Pruning Model Validation Accuracy: {acc:.2f}%")

latency, throughput, power, energy, edp = measure_inference_metrics(pruned_model,
                                                                        val_loader)
flops, params = measure_model_size_and_flops(pruned_model, input_res=(1, 3, 224, 224))

print("\nInference Metrics:")
print(f"Total inference time: {latency * 10000:.2f} s")
print(f"Total images processed: 10000")
print(f"Average latency per image: {latency * 1000:.2f} ms")
print(f"Throughput: {throughput:.2f} images/s")
print(f"Average GPU Power: {power:.2f} W")
print(f"Energy per image: {energy:.4f} J")
print(f"Energy-Delay Product (EDP): {edp:.6f} J*s")

print("\nModel Size and FLOPs:")
print(f"FLOPs: {flops:.2f} GFLOPs")
print(f"Number of parameters: {params / 1e6:.2f} million")